# Phase 4 v4: DeBERTa LoRA + Qwen QLoRA — Beat RESTBERTa-PM (0.8529)

**Key fixes in this version:**
1. Training tokenization uses `return_overflowing_tokens=True` — matches baseline methodology (creates natural 84/16 non-answerable/answerable distribution)
2. Evaluation uses overflow tokens too — fair comparison with baseline
3. Skip-if-done logic everywhere — loads existing results from Drive if present
4. Robust Drive handling with auto-remount
5. Qwen Part B uses few-shot prompting for better extraction
6. Full merged model saved to Drive (no MISSING qa_outputs bug)

**Run order:** Run cells top to bottom. Will auto-skip completed steps.

In [ ]:
# Cell 1: Drive mount + auth (resilient)
import os

def ensure_drive():
    if os.path.exists('/content/drive/MyDrive'):
        print('Drive already mounted.')
        return
    from google.colab import drive
    try:
        drive.mount('/content/drive')
    except Exception as e:
        print(f'Mount failed: {e}, trying force remount...')
        try:
            drive.flush_and_unmount()
        except Exception:
            pass
        drive.mount('/content/drive', force_remount=True)
    print('Drive mounted.')

ensure_drive()

In [ ]:
# Cell 2: Install deps (idempotent)
!pip install -q peft>=0.13.0 bitsandbytes>=0.43.0 accelerate>=0.34.0 trl>=0.12.0 transformers>=4.45.0 datasets psutil 2>&1 | tail -2
print('Deps installed.')

In [ ]:
# Cell 3: Download dataset to Drive (one-time)
import os, shutil

DRIVE_BASE = '/content/drive/MyDrive/restberta'
DRIVE_DATA = os.path.join(DRIVE_BASE, 'restberta-datasets')
os.makedirs(DRIVE_BASE, exist_ok=True)

if not os.path.exists(f'{DRIVE_DATA}/parameter_matching/train/train.json'):
    print('Downloading dataset from Kaggle...')
    # EDIT THESE with your Kaggle credentials
    os.environ['KAGGLE_USERNAME'] = 'YOUR_KAGGLE_USERNAME'
    os.environ['KAGGLE_KEY'] = 'YOUR_KAGGLE_KEY'
    !pip install -q kaggle
    !kaggle datasets download -d kailram/restberta-datasets -p /tmp/restberta --unzip
    shutil.copytree('/tmp/restberta', DRIVE_DATA, dirs_exist_ok=True)
    !rm -rf /tmp/restberta
    print('Dataset downloaded.')
else:
    print('Dataset already on Drive.')

# Verify
for split in ['parameter_matching/train', 'parameter_matching/validation',
              'endpoint_discovery/train', 'endpoint_discovery/validation']:
    p = os.path.join(DRIVE_DATA, split)
    n = len(os.listdir(p)) if os.path.isdir(p) else 0
    print(f"  {'OK' if n else 'MISSING':>7}  {split}/ ({n} files)")

In [ ]:
# Cell 4: Imports + Config
import os, json, gc, glob, time, random, re, shutil, psutil
import numpy as np
import torch
from datetime import datetime
from tqdm import tqdm

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

DRIVE_BASE  = '/content/drive/MyDrive/restberta'
DRIVE_DATA  = os.path.join(DRIVE_BASE, 'restberta-datasets')
PM_PATH     = os.path.join(DRIVE_DATA, 'parameter_matching')
ED_PATH     = os.path.join(DRIVE_DATA, 'endpoint_discovery')
RESULTS_DIR = os.path.join(DRIVE_BASE, 'results_phase4_v4')
CKPT_DIR    = os.path.join(DRIVE_BASE, 'checkpoints_phase4_v4')
CACHE_DIR   = os.path.join(DRIVE_BASE, 'cache_phase4_v4')
for d in [RESULTS_DIR, CKPT_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

MAX_LENGTH   = 512
DOC_STRIDE   = 128
N_BEST       = 10
TOP_K        = list(range(1, 11))
SEED         = 42
MAX_TRAIN_QA = 8_000      # Reduced! 8K × overflow ≈ 50K features
EVAL_BATCH   = 4
TRAIN_BATCH  = 2
GRAD_ACCUM   = 16
EPOCHS       = 1
LR           = 3e-4

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda', 'No GPU!'

def print_ram(tag=''):
    r = psutil.virtual_memory()
    g = torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0
    print(f'  [{tag}] RAM: {r.used/1e9:.1f}/{r.total/1e9:.1f} GB ({r.percent}%) | GPU: {g:.1f} GB')

def result_exists(name, ds_tag):
    matches = glob.glob(os.path.join(RESULTS_DIR, f'{name}_{ds_tag}*.json'))
    return matches[0] if matches else None

def load_existing_result(path):
    with open(path) as f: return json.load(f)

def save_result(result, name, ds_tag):
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    path = os.path.join(RESULTS_DIR, f'{name}_{ds_tag}_{ts}.json')
    with open(path, 'w') as f: json.dump(result, f, indent=2)
    print(f'  Saved: {os.path.basename(path)}')
    return path

print(f'GPU: {torch.cuda.get_device_name(0)}')
print_ram('init')
print(f'Config: {MAX_TRAIN_QA} QA pairs × overflow, {EPOCHS} epoch')

In [ ]:
# Cell 5: Shared functions — data reading, property parsing

def read_qa_pairs_sampled(data_path, split, max_qa):
    '''Reservoir sampling — streams 909K without loading into RAM.'''
    reservoir = []
    count = 0
    for fpath in sorted(glob.glob(os.path.join(data_path, split, '*.json'))):
        with open(fpath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                rec = json.loads(line)
                ctx = rec['context']
                for q in rec['questions']:
                    count += 1
                    item = (ctx, q['question'], q['answers'])
                    if len(reservoir) < max_qa:
                        reservoir.append(item)
                    else:
                        j = random.randint(0, count - 1)
                        if j < max_qa:
                            reservoir[j] = item
    print(f'  Scanned {count}, kept {len(reservoir)}')
    random.shuffle(reservoir)
    return reservoir


def read_qa_pairs_validation(data_path):
    qa_pairs = []
    for fpath in sorted(glob.glob(os.path.join(data_path, 'validation', '*.json'))):
        with open(fpath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                rec = json.loads(line)
                ctx = rec['context']
                for q in rec['questions']:
                    qa_pairs.append((ctx, q['question'], q['answers']))
    return qa_pairs


def precompute_props(ctx):
    '''Parse context into list of (start, end, name) space-separated properties.'''
    props = []
    i, n = 0, len(ctx)
    while i < n:
        while i < n and ctx[i] == ' ': i += 1
        if i >= n: break
        start = i
        while i < n and ctx[i] != ' ': i += 1
        props.append((start, i, ctx[start:i]))
    return props


def determine_best_from_span(props, char_s, char_e):
    '''Match baseline algorithm: full match > longest unique partial.'''
    full = None
    partials = []
    for ps, pe, name in props:
        if char_e <= ps or char_s >= pe: continue
        is_full = (char_s <= ps and char_e >= pe)
        if is_full:
            if full is not None: return None
            full = name
        else:
            partials.append((pe - ps, name))
    if full is not None: return full
    if not partials: return None
    partials.sort(key=lambda x: -x[0])
    if len(partials) >= 2 and partials[0][0] == partials[1][0]: return None
    return partials[0][1]


print('Shared functions loaded.')
print_ram('funcs')

---
## Part A: DeBERTa-v3-large + LoRA (Primary)

**Goal**: Beat 0.8529 Acc@1 using LoRA on DeBERTa-v3-large encoder.

Training approach: overflow tokenization creates ~6x features per QA pair, with natural ~84/16 non-answerable/answerable distribution. Model learns both to extract spans AND predict CLS when answer isn't in window.

In [ ]:
# Cell 6: TOKENIZE training data with OVERFLOW + max 3 non-answer windows/pair
# Matches baseline's max_no_answers_per_possible_answer=3

DEBERTA_CKPT = 'microsoft/deberta-v3-large'
DEBERTA_NAME = 'deberta_v3_large_lora'
cache_path = os.path.join(CACHE_DIR, f'{DEBERTA_NAME}_train_overflow_v2.npz')
MAX_NON_ANS_PER_PAIR = 3  # Baseline setting

if os.path.exists(cache_path):
    sz = os.path.getsize(cache_path)/1e6
    print(f'Cache exists ({sz:.0f} MB) — skip to training cell.')
else:
    print('Step 1: Sample training data...')
    print_ram('before-sample')
    train_data = read_qa_pairs_sampled(PM_PATH, 'train', MAX_TRAIN_QA)
    ed = read_qa_pairs_sampled(ED_PATH, 'train', min(1000, MAX_TRAIN_QA//10))
    train_data.extend(ed); del ed
    random.shuffle(train_data)
    print(f'Total: {len(train_data)} QA pairs')
    print_ram('after-sample')

    print('\nStep 2: Tokenize with OVERFLOW + subsample non-answer windows...')
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(DEBERTA_CKPT)
    pad_on_right = tok.padding_side == 'right'
    ctx_tok_idx = 1 if pad_on_right else 0

    batch_ids, batch_masks, batch_starts, batch_ends = [], [], [], []
    stats = {'answerable_feats': 0, 'unanswerable_feats': 0, 'non_ans_dropped': 0}

    BATCH = 30
    for b0 in tqdm(range(0, len(train_data), BATCH), desc='  Tokenizing'):
        batch = train_data[b0:b0+BATCH]
        ctxs = [x[0] for x in batch]
        ques = [x[1] for x in batch]

        tokenized = tok(
            ques if pad_on_right else ctxs,
            ctxs if pad_on_right else ques,
            truncation='only_second' if pad_on_right else 'only_first',
            max_length=MAX_LENGTH,
            stride=DOC_STRIDE,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            padding='max_length',
        )
        sm = tokenized.pop('overflow_to_sample_mapping')
        offsets_list = tokenized.pop('offset_mapping')

        N = len(tokenized['input_ids'])

        # Phase 1: Determine start/end for each feature + group by QA pair
        per_feat = []  # list of (qa_idx, sp, ep, is_answer)
        for fi in range(N):
            qa_idx = sm[fi]
            _, _, answers = batch[qa_idx]
            seq_ids = tokenized.sequence_ids(fi)
            cp = 0

            if not answers['answer_start']:
                sp = ep = cp
                is_ans = False
            else:
                sc = answers['answer_start'][0]
                ec = sc + len(answers['text'][0])
                ts = 0
                while ts < MAX_LENGTH and seq_ids[ts] != ctx_tok_idx: ts += 1
                te = MAX_LENGTH - 1
                while te >= 0 and seq_ids[te] != ctx_tok_idx: te -= 1
                if ts >= MAX_LENGTH or te < 0:
                    sp = ep = cp
                    is_ans = False
                elif offsets_list[fi][ts][0] > sc or offsets_list[fi][te][1] < ec:
                    sp = ep = cp
                    is_ans = False
                else:
                    while ts < MAX_LENGTH and offsets_list[fi][ts][0] <= sc: ts += 1
                    sp = max(ts - 1, 0)
                    while te >= 0 and offsets_list[fi][te][1] >= ec: te -= 1
                    ep = min(te + 1, MAX_LENGTH - 1)
                    if sp > ep or sp < 0 or ep >= MAX_LENGTH:
                        sp = ep = cp
                        is_ans = False
                    else:
                        is_ans = True
            per_feat.append((qa_idx, sp, ep, is_ans))

        # Phase 2: Subsample non-answer windows per QA pair (max 3)
        from collections import defaultdict
        by_pair = defaultdict(lambda: {'ans': [], 'non': []})
        for fi, (qa_idx, sp, ep, is_ans) in enumerate(per_feat):
            if is_ans:
                by_pair[qa_idx]['ans'].append(fi)
            else:
                by_pair[qa_idx]['non'].append(fi)

        keep_indices = []
        for qa_idx, groups in by_pair.items():
            keep_indices.extend(groups['ans'])
            # Cap non-answer windows at MAX_NON_ANS_PER_PAIR
            non_ans = groups['non']
            if len(non_ans) > MAX_NON_ANS_PER_PAIR:
                random.shuffle(non_ans)
                dropped = len(non_ans) - MAX_NON_ANS_PER_PAIR
                stats['non_ans_dropped'] += dropped
                non_ans = non_ans[:MAX_NON_ANS_PER_PAIR]
            keep_indices.extend(non_ans)
        keep_indices.sort()

        # Build numpy arrays from kept indices only
        K = len(keep_indices)
        ids_np = np.zeros((K, MAX_LENGTH), dtype=np.int32)
        masks_np = np.zeros((K, MAX_LENGTH), dtype=np.int32)
        starts_np = np.zeros(K, dtype=np.int64)
        ends_np = np.zeros(K, dtype=np.int64)

        for out_idx, fi in enumerate(keep_indices):
            ids_np[out_idx] = tokenized['input_ids'][fi]
            masks_np[out_idx] = tokenized['attention_mask'][fi]
            _, sp, ep, is_ans = per_feat[fi]
            starts_np[out_idx] = sp
            ends_np[out_idx] = ep
            if is_ans: stats['answerable_feats'] += 1
            else: stats['unanswerable_feats'] += 1

        batch_ids.append(ids_np)
        batch_masks.append(masks_np)
        batch_starts.append(starts_np)
        batch_ends.append(ends_np)
        del tokenized, offsets_list, sm, ctxs, ques, batch, per_feat, by_pair, keep_indices

        if (b0 // BATCH) % 50 == 0:
            gc.collect()

    print(f'\nStats:')
    print(f'  Answerable:      {stats["answerable_feats"]}')
    print(f'  Unanswerable:    {stats["unanswerable_feats"]}')
    print(f'  Dropped non-ans: {stats["non_ans_dropped"]}')
    total = stats['answerable_feats'] + stats['unanswerable_feats']
    print(f'  Total features:  {total} (ans ratio: {stats["answerable_feats"]/total:.2%})')
    print_ram('after-tokenize')

    print('\nConcatenating...')
    all_ids = np.concatenate(batch_ids); del batch_ids
    all_masks = np.concatenate(batch_masks); del batch_masks
    all_starts = np.concatenate(batch_starts); del batch_starts
    all_ends = np.concatenate(batch_ends); del batch_ends
    gc.collect()

    print(f'  Shape: {all_ids.shape}')
    print_ram('after-concat')

    np.savez_compressed(cache_path, ids=all_ids, masks=all_masks,
                        starts=all_starts, ends=all_ends)
    print(f'Saved: {cache_path} ({os.path.getsize(cache_path)/1e6:.0f} MB)')

    del all_ids, all_masks, all_starts, all_ends, train_data, tok
    gc.collect()
    print_ram('after-free')
    print('\n>>> RESTART RUNTIME, then run cells 1→4→5→8 <<<')

In [ ]:
# Cell 7: Train DeBERTa-v3-large + LoRA (skip if merged model exists)
try: print_ram
except NameError: raise SystemExit('Re-run Cell 4 first!')

from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

DEBERTA_CKPT = 'microsoft/deberta-v3-large'
DEBERTA_NAME = 'deberta_v3_large_lora'
merged_path = os.path.join(CKPT_DIR, f'{DEBERTA_NAME}_merged_v3')
cache_path = os.path.join(CACHE_DIR, f'{DEBERTA_NAME}_train_overflow_v2.npz')

class NpzDataset(Dataset):
    def __init__(self, path):
        d = np.load(path)
        self.ids = d['ids'].astype(np.int64)
        self.masks = d['masks'].astype(np.int64)
        self.starts = d['starts'].astype(np.int64)
        self.ends = d['ends'].astype(np.int64)
        np.clip(self.starts, 0, self.ids.shape[1]-1, out=self.starts)
        np.clip(self.ends, 0, self.ids.shape[1]-1, out=self.ends)
        n_ans = int(np.sum(self.starts != 0))
        print(f'  {len(self.ids)} features | answerable: {n_ans} ({100*n_ans/len(self.ids):.0f}%)')
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        return {
            'input_ids': torch.from_numpy(self.ids[i].copy()),
            'attention_mask': torch.from_numpy(self.masks[i].copy()),
            'start_positions': torch.tensor(self.starts[i]),
            'end_positions': torch.tensor(self.ends[i]),
        }

# SKIP if already trained
if os.path.exists(os.path.join(merged_path, 'config.json')):
    print(f'SKIP: Merged model already exists at {merged_path}')
    print(f'  Files: {os.listdir(merged_path)[:6]}')
elif not os.path.exists(cache_path):
    raise SystemExit(f'Run Cell 6 first to create tokenization cache')
else:
    print_ram('before-train')
    train_ds = NpzDataset(cache_path)

    model = AutoModelForQuestionAnswering.from_pretrained(DEBERTA_CKPT)

    # Auto-detect LoRA targets
    linear_names = set()
    for name, mod in model.named_modules():
        if hasattr(mod, 'weight') and len(mod.weight.shape) == 2:
            linear_names.add(name.split('.')[-1])
    targets = [n for n in ['query_proj','key_proj','value_proj'] if n in linear_names]
    if not targets: targets = list(linear_names)
    print(f'  LoRA targets: {targets}')

    peft_config = LoraConfig(
        task_type=TaskType.QUESTION_ANS,
        r=32, lora_alpha=32, lora_dropout=0.05,
        target_modules=targets,
        modules_to_save=['qa_outputs'],   # save QA head with adapter
    )
    model = get_peft_model(model, peft_config)

    # FP16 fix: cast trainable params to float32
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()

    model.print_trainable_parameters()
    print_ram('model-loaded')

    n_steps = len(train_ds) // (TRAIN_BATCH * GRAD_ACCUM)
    print(f'  Steps: ~{n_steps} | Effective batch: {TRAIN_BATCH*GRAD_ACCUM}')

    args = TrainingArguments(
        output_dir='/tmp/deberta_tmp',
        per_device_train_batch_size=TRAIN_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=2,    # 2 epochs OK with safer config
        learning_rate=2e-4,        # lowered from 3e-4 to prevent collapse
        lr_scheduler_type='cosine',
        warmup_ratio=0.06,
        weight_decay=0.01,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        logging_steps=100,
        save_strategy='no',
        report_to='none',
        dataloader_num_workers=0,
        dataloader_pin_memory=False,
        remove_unused_columns=False,
    )

    trainer = Trainer(model=model, args=args, train_dataset=train_ds)

    # Test forward pass
    model.train()
    batch = {k: v.unsqueeze(0).to(device) for k, v in train_ds[0].items()}
    with torch.amp.autocast('cuda', dtype=torch.float16):
        out = model(**batch)
    print(f'  Test loss: {out.loss.item():.4f}')
    del batch, out

    print('\nTraining...')
    t0 = time.time()
    trainer.train()
    print(f'Done in {(time.time()-t0)/60:.1f} min')

    # Merge LoRA into base, save full model
    print('\nMerging + saving full model...')
    merged = model.merge_and_unload()

    qa_w = merged.qa_outputs.weight
    print(f'  QA head: {tuple(qa_w.shape)} mean={qa_w.mean():.4f} std={qa_w.std():.4f}')
    assert qa_w.std() > 0.001, 'QA head untrained!'

    if os.path.exists(merged_path):
        shutil.rmtree(merged_path)
    os.makedirs(merged_path)
    merged.save_pretrained(merged_path, safe_serialization=True)
    AutoTokenizer.from_pretrained(DEBERTA_CKPT).save_pretrained(merged_path)

    files = os.listdir(merged_path)
    print(f'  Files saved: {files}')
    assert any('model.safetensors' in f or 'pytorch_model' in f for f in files)

    del model, merged, trainer, train_ds
    gc.collect(); torch.cuda.empty_cache()
    print_ram('after-train')

In [ ]:
# Better sanity check — uses null score vs best span (like real eval)
import torch, numpy as np, glob, json

DEBERTA_NAME = 'deberta_v3_large_lora'
merged_path = "/content/drive/MyDrive/restberta/checkpoints_phase4_v4/deberta_v3_large_lora_merged_v2"
PM_PATH = "/content/drive/MyDrive/restberta/restberta-datasets/parameter_matching"

from transformers import AutoTokenizer, AutoModelForQuestionAnswering
print('Loading model...')
model = AutoModelForQuestionAnswering.from_pretrained(merged_path)
model.to(device).half().eval()
eval_tok = AutoTokenizer.from_pretrained(merged_path)
print(f'QA head std: {model.qa_outputs.weight.std():.4f}')

# Load samples
qa_pairs_short, qa_pairs_long = [], []
with open(f'{PM_PATH}/validation/validation.json') as f:
    for line in f:
        rec = json.loads(line.strip())
        for q in rec['questions']:
            if not q['answers']['answer_start']: continue
            ctx_len = len(rec['context'])
            ans_pos = q['answers']['answer_start'][0]
            if ans_pos < 1500 and ctx_len < 4000:  # short context
                if len(qa_pairs_short) < 30:
                    qa_pairs_short.append((rec['context'], q['question'], q['answers']))
            elif ans_pos > 8000:  # answer truly beyond 512 tokens
                if len(qa_pairs_long) < 30:
                    qa_pairs_long.append((rec['context'], q['question'], q['answers']))
        if len(qa_pairs_short) >= 30 and len(qa_pairs_long) >= 30: break

print(f'\nTesting {len(qa_pairs_short)} short + {len(qa_pairs_long)} long')

def smart_predict(ctx, q, ans):
    inputs = eval_tok(q, ctx, truncation='only_second', max_length=512,
                      padding='max_length', return_tensors='pt').to(device)
    with torch.no_grad():
        out = model(**inputs)
    sl = out.start_logits[0].cpu().float().numpy()
    el = out.end_logits[0].cpu().float().numpy()
    
    # Null score = score at CLS (position 0)
    null_score = sl[0] + el[0]
    
    # Best span score (excluding CLS)
    seq_ids = inputs.sequence_ids(0)
    ctx_mask = np.array([s == 1 for s in seq_ids], dtype=bool)
    
    best_span_score = -float('inf')
    best_pred = ''
    for si in np.argsort(sl)[-10:][::-1]:
        if si == 0 or not ctx_mask[si]: continue
        for ei in np.argsort(el)[-10:][::-1]:
            if ei == 0 or not ctx_mask[ei] or ei < si or ei - si > 30: continue
            score = sl[si] + el[ei]
            if score > best_span_score:
                best_span_score = score
                best_pred = eval_tok.decode(inputs['input_ids'][0][si:ei+1], skip_special_tokens=True).strip()
    
    # Predict null if null_score > best_span_score
    is_null = null_score > best_span_score
    return best_pred, null_score, best_span_score, is_null

# Test short-context (should extract correctly)
correct_short = 0
print('\nShort-context samples (model should extract):')
for ctx, q, ans in qa_pairs_short[:5]:
    pred, ns, bs, is_null = smart_predict(ctx, q, ans)
    gt = ans['text'][0].strip()
    match = pred == gt or gt in pred or pred in gt
    if match: correct_short += 1
    print(f"  {'✓' if match else '✗'} gt={gt[:25]:25} pred={pred[:25]:25} null={ns:.1f} span={bs:.1f}")

for ctx, q, ans in qa_pairs_short[5:]:
    pred, ns, bs, is_null = smart_predict(ctx, q, ans)
    gt = ans['text'][0].strip()
    if pred == gt or gt in pred or pred in gt: correct_short += 1

# Test long-context (should predict null)
correct_long = 0
print('\nLong-context samples (model should predict null):')
for ctx, q, ans in qa_pairs_long[:5]:
    pred, ns, bs, is_null = smart_predict(ctx, q, ans)
    if is_null: correct_long += 1
    print(f"  {'✓' if is_null else '✗'} null_pred={is_null}  null={ns:.1f} span={bs:.1f}  diff={ns-bs:+.1f}")

for ctx, q, ans in qa_pairs_long[5:]:
    _, ns, bs, is_null = smart_predict(ctx, q, ans)
    if is_null: correct_long += 1

print(f'\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'Short-context: {correct_short}/30 correct ({100*correct_short/30:.0f}%)')
print(f'Long-context:  {correct_long}/30 predict null ({100*correct_long/30:.0f}%)')
print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')

# CLS bias diagnostic
cls_scores = []
span_scores = []
for ctx, q, ans in qa_pairs_long[:30]:
    _, ns, bs, _ = smart_predict(ctx, q, ans)
    cls_scores.append(ns)
    span_scores.append(bs)

avg_diff = np.mean(np.array(cls_scores) - np.array(span_scores))
print(f'\nDIAGNOSTIC: avg(null_score - best_span_score) on long contexts: {avg_diff:+.2f}')
print(f'  Negative → model UNDER-predicts null. Need bias of ~+{abs(avg_diff):.1f} to fix.')
print(f'  Positive → model already predicts null correctly.')


In [ ]:
# Cell 9: Full eval with OVERFLOW tokens (matches baseline methodology)

def overflow_eval(data_path, ds_name, batch_size=EVAL_BATCH, max_length=MAX_LENGTH, doc_stride=DOC_STRIDE, n_best=N_BEST):
    print(f'\n{"="*60}\n  EVAL: {DEBERTA_NAME} on {ds_name}\n{"="*60}')

    qa_pairs = read_qa_pairs_validation(data_path)
    print(f'  {len(qa_pairs)} QA pairs')

    results = [{'rank': k, 'correct': 0, 'correct_answerable': 0, 'correct_non_answerable': 0,
                'total': 0, 'total_answerable': 0, 'total_non_answerable': 0} for k in TOP_K]

    t0 = time.time()
    model.eval()
    feat_count = 0

    for b0 in tqdm(range(0, len(qa_pairs), batch_size), desc='  Eval'):
        batch = qa_pairs[b0:b0+batch_size]
        ctxs = [x[0] for x in batch]
        ques = [x[1] for x in batch]
        answers_list = [x[2] for x in batch]

        enc = eval_tok(
            ques, ctxs,
            truncation='only_second',
            max_length=max_length,
            stride=doc_stride,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            padding='max_length',
            return_tensors='pt',
        )
        sample_mapping = enc.pop('overflow_to_sample_mapping').tolist()
        offsets = enc.pop('offset_mapping').numpy()
        seq_ids_list = [enc.sequence_ids(i) for i in range(len(enc['input_ids']))]
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            out = model(**enc)

        F = len(sample_mapping)
        feat_count += F

        top_s_vals, top_s_idx = torch.topk(out.start_logits, n_best, dim=-1)
        top_e_vals, top_e_idx = torch.topk(out.end_logits, n_best, dim=-1)
        top_s_idx = top_s_idx.cpu().numpy()
        top_e_idx = top_e_idx.cpu().numpy()
        top_s_vals = top_s_vals.cpu().numpy()
        top_e_vals = top_e_vals.cpu().numpy()
        cls_s = out.start_logits[:, 0].cpu().numpy()
        cls_e = out.end_logits[:, 0].cpu().numpy()

        # Cache props per unique context in this batch
        props_cache = {}

        for fi in range(F):
            sample_idx = sample_mapping[fi]
            ctx_text = ctxs[sample_idx]
            ans = answers_list[sample_idx]
            gt_name = ans['text'][0] if ans['text'] else ''

            if id(ctx_text) not in props_cache:
                props_cache[id(ctx_text)] = precompute_props(ctx_text)
            props = props_cache[id(ctx_text)]

            seq_ids = seq_ids_list[fi]
            offs = offsets[fi]

            # Determine if this feature contains the answer
            if not ans['answer_start']:
                is_oos = True
            else:
                sc = ans['answer_start'][0]
                ec = sc + len(ans['text'][0])
                ctx_positions = [p for p, s in enumerate(seq_ids) if s == 1]
                if not ctx_positions:
                    is_oos = True
                else:
                    first_char = int(offs[ctx_positions[0], 0])
                    last_char = int(offs[ctx_positions[-1], 1])
                    is_oos = not (first_char <= sc and last_char >= ec)

            ctx_mask = np.array([s == 1 for s in seq_ids], dtype=bool)
            candidates = []

            for si in range(n_best):
                s = int(top_s_idx[fi, si])
                if s == 0 or not ctx_mask[s]: continue
                for ei in range(n_best):
                    e = int(top_e_idx[fi, ei])
                    if e == 0 or not ctx_mask[e]: continue
                    if e < s: continue
                    if e - s > 30: continue

                    char_s = int(offs[s, 0])
                    char_e = int(offs[e, 1])
                    name = determine_best_from_span(props, char_s, char_e)
                    if name is None: continue

                    score = float(top_s_vals[fi, si] + top_e_vals[fi, ei])
                    candidates.append((score, name))

            # CLS = "no answer" candidate
            cls_score = float(cls_s[fi] + cls_e[fi])
            candidates.append((cls_score, None))
            candidates.sort(key=lambda x: -x[0])

            rank = None
            for ri, (_, name) in enumerate(candidates):
                if is_oos:
                    if name is None or name == gt_name:
                        rank = ri + 1; break
                else:
                    if name is not None and name == gt_name:
                        rank = ri + 1; break

            for kr in results:
                if rank is not None and rank <= kr['rank']:
                    kr['correct'] += 1
                    if is_oos: kr['correct_non_answerable'] += 1
                    else: kr['correct_answerable'] += 1
                kr['total'] += 1
                if is_oos: kr['total_non_answerable'] += 1
                else: kr['total_answerable'] += 1

    for kr in results:
        t = kr['total']; ta = kr['total_answerable']; tn = kr['total_non_answerable']
        kr['accuracy'] = kr['correct']/t if t else 0
        kr['accuracy_answerable'] = kr['correct_answerable']/ta if ta else 0
        kr['accuracy_non_answerable'] = kr['correct_non_answerable']/tn if tn else 0

    elapsed = time.time() - t0
    print(f'\n  Done: {elapsed/60:.1f} min | {feat_count} features ({feat_count/len(qa_pairs):.1f}x QA)')
    print(f'  Answerable: {results[0]["total_answerable"]} | Non-ans: {results[0]["total_non_answerable"]}')
    print(f'  K   Acc@K   Ans     Non')
    for r in results:
        print(f"  @{r['rank']:>2}  {r['accuracy']:.4f}  {r['accuracy_answerable']:.4f}  {r['accuracy_non_answerable']:.4f}")

    return {
        'model': DEBERTA_NAME, 'dataset': ds_name, 'type': 'fine-tuned',
        'total_samples': results[0]['total'],
        'total_answerable': results[0]['total_answerable'],
        'total_non_answerable': results[0]['total_non_answerable'],
        'elapsed_seconds': elapsed, 'results': results,
    }


# Check existing results first
pm_done = result_exists(DEBERTA_NAME, 'pm')
ed_done = result_exists(DEBERTA_NAME, 'ed')

all_results_phase4 = []

if pm_done:
    print(f'Loading existing PM result: {os.path.basename(pm_done)}')
    all_results_phase4.append(load_existing_result(pm_done))
else:
    r = overflow_eval(PM_PATH, 'parameter_matching')
    save_result(r, DEBERTA_NAME, 'pm')
    all_results_phase4.append(r)

if ed_done:
    print(f'Loading existing ED result: {os.path.basename(ed_done)}')
    all_results_phase4.append(load_existing_result(ed_done))
else:
    r = overflow_eval(ED_PATH, 'endpoint_discovery')
    save_result(r, DEBERTA_NAME, 'ed')
    all_results_phase4.append(r)

print(f'\nDeBERTa Part A: {len(all_results_phase4)} results')

---
## Part B: Qwen2.5-3B + QLoRA (Decoder with Few-Shot Prompting)

Uses **few-shot prompting** with 3 example QA pairs + structured output format for better extraction. This is an alternative to LlamaIndex that works better for span extraction tasks.

Skip-if-done: will use cached results if already run.

In [ ]:
# Cell 10: Qwen QLoRA — train with few-shot prompting
import os, glob, json

QWEN_CKPT = 'Qwen/Qwen2.5-3B-Instruct'
QWEN_NAME = 'qwen25_3b_qlora'
QWEN_MAX_TRAIN = 10_000
qwen_merged_path = os.path.join(CKPT_DIR, f'{QWEN_NAME}_merged')

# Skip if merged model exists
if os.path.exists(os.path.join(qwen_merged_path, 'config.json')):
    print(f'SKIP: Qwen already trained at {qwen_merged_path}')
else:
    print('Preparing Qwen QLoRA training data...')

    # Free GPU from DeBERTa
    try:
        del model, eval_tok
    except NameError:
        pass
    gc.collect(); torch.cuda.empty_cache()

    # Sample training data (balanced)
    reservoir_ans, reservoir_non = [], []
    half = QWEN_MAX_TRAIN // 2
    count = 0
    for fp in sorted(glob.glob(f'{PM_PATH}/train/*.json')):
        with open(fp) as f:
            for line in f:
                rec = json.loads(line.strip())
                ctx = rec['context']
                for q in rec['questions']:
                    count += 1
                    item = (ctx, q['question'], q['answers'])
                    if q['answers']['answer_start']:
                        if len(reservoir_ans) < half:
                            reservoir_ans.append(item)
                        else:
                            j = random.randint(0, count)
                            if j < half: reservoir_ans[j] = item
                    else:
                        if len(reservoir_non) < half:
                            reservoir_non.append(item)
                        else:
                            j = random.randint(0, count)
                            if j < half: reservoir_non[j] = item
    train_data = reservoir_ans + reservoir_non
    random.shuffle(train_data)
    print(f'  Sampled {len(train_data)} ({len(reservoir_ans)} ans + {len(reservoir_non)} non)')

    # Few-shot examples (static)
    FEWSHOT = [
        {
            'schema': 'users[*].id users[*].name users[*].email users[*].created_at',
            'desc': 'The user\'s email address',
            'answer': 'users[*].email'
        },
        {
            'schema': 'products[*].sku products[*].name products[*].price products[*].inventory.count',
            'desc': 'The number of items currently in stock',
            'answer': 'products[*].inventory.count'
        },
        {
            'schema': 'order.id order.total order.items[*].quantity',
            'desc': 'The shipping address of the order',
            'answer': 'NO_ANSWER'
        },
    ]

    SYSTEM_MSG = ('You are an API documentation expert. Given a schema and description, '
                  'extract the EXACT matching parameter name from the schema. '
                  'Return ONLY the exact text from the schema. If no match, return NO_ANSWER.')

    def build_fewshot_prompt(ctx, question, ans_text):
        messages = [{'role': 'system', 'content': SYSTEM_MSG}]
        for ex in FEWSHOT:
            messages.append({'role': 'user', 'content': f'Schema:\n{ex["schema"]}\n\nDescription: {ex["desc"]}'})
            messages.append({'role': 'assistant', 'content': ex['answer']})
        messages.append({'role': 'user', 'content': f'Schema:\n{ctx[:2500]}\n\nDescription: {question}'})
        messages.append({'role': 'assistant', 'content': ans_text})
        return messages

    chat_data = []
    for ctx, q, a in train_data:
        ans_text = a['text'][0] if a['text'] and a['answer_start'] else 'NO_ANSWER'
        chat_data.append(build_fewshot_prompt(ctx, q, ans_text))
    del train_data; gc.collect()

    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, prepare_model_for_kbit_training, PeftModel
    from trl import SFTTrainer, SFTConfig
    from datasets import Dataset as HFDataset

    print('\nLoading Qwen2.5-3B in 4-bit...')
    qwen_tok = AutoTokenizer.from_pretrained(QWEN_CKPT, trust_remote_code=True)
    if qwen_tok.pad_token is None:
        qwen_tok.pad_token = qwen_tok.eos_token
    qwen_tok.padding_side = 'right'

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    qwen_model = AutoModelForCausalLM.from_pretrained(
        QWEN_CKPT, quantization_config=bnb_config,
        device_map='auto', trust_remote_code=True,
    )
    qwen_model = prepare_model_for_kbit_training(qwen_model)
    print_ram('qwen-loaded')

    qlora_config = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        task_type='CAUSAL_LM',
    )

    texts = [qwen_tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=False) for chat in chat_data]
    sft_dataset = HFDataset.from_dict({'text': texts})
    del chat_data, texts; gc.collect()
    print(f'SFT dataset: {len(sft_dataset)}')

    sft_args = SFTConfig(
        output_dir='/tmp/qwen_tmp',
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        logging_steps=50,
        save_strategy='no',
        report_to='none',
        dataloader_num_workers=0,
        max_length=1536,  # larger for few-shot
        packing=False,
    )

    trainer = SFTTrainer(
        model=qwen_model, train_dataset=sft_dataset,
        args=sft_args, peft_config=qlora_config,
        processing_class=qwen_tok,
    )

    print('\nTraining Qwen QLoRA...')
    t0 = time.time()
    trainer.train()
    print(f'Done in {(time.time()-t0)/60:.1f} min')

    # Save adapter, reload without quantization, merge, save
    adapter_path = '/tmp/qwen_adapter'
    trainer.model.save_pretrained(adapter_path)
    del qwen_model, trainer; gc.collect(); torch.cuda.empty_cache()

    base = AutoModelForCausalLM.from_pretrained(
        QWEN_CKPT, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
    peft_model = PeftModel.from_pretrained(base, adapter_path)
    merged = peft_model.merge_and_unload()

    if os.path.exists(qwen_merged_path):
        shutil.rmtree(qwen_merged_path)
    os.makedirs(qwen_merged_path)
    merged.save_pretrained(qwen_merged_path, safe_serialization=True)
    qwen_tok.save_pretrained(qwen_merged_path)
    print(f'\nSaved to {qwen_merged_path}')

    del merged, peft_model, base; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 11: Qwen eval (2K subset) with few-shot prompting
QWEN_EVAL_SUBSET = 2000
QWEN_NAME = 'qwen25_3b_qlora'
qwen_merged_path = os.path.join(CKPT_DIR, f'{QWEN_NAME}_merged')

qwen_pm_done = result_exists(QWEN_NAME, 'pm')
qwen_ed_done = result_exists(QWEN_NAME, 'ed')

if qwen_pm_done and qwen_ed_done:
    print(f'SKIP: Qwen results already exist')
    all_results_phase4.append(load_existing_result(qwen_pm_done))
    all_results_phase4.append(load_existing_result(qwen_ed_done))
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM

    print('Loading Qwen merged model...')
    qwen_tok = AutoTokenizer.from_pretrained(qwen_merged_path, trust_remote_code=True)
    if qwen_tok.pad_token is None: qwen_tok.pad_token = qwen_tok.eos_token
    qwen_model = AutoModelForCausalLM.from_pretrained(
        qwen_merged_path, torch_dtype=torch.float16,
        device_map='auto', trust_remote_code=True)
    qwen_model.eval()
    print_ram('qwen-eval-loaded')

    SYSTEM_MSG = ('You are an API documentation expert. Given a schema and description, '
                  'extract the EXACT matching parameter name from the schema. '
                  'Return ONLY the exact text from the schema. If no match, return NO_ANSWER.')
    FEWSHOT = [
        {'schema': 'users[*].id users[*].name users[*].email users[*].created_at',
         'desc': "The user's email address", 'answer': 'users[*].email'},
        {'schema': 'products[*].sku products[*].name products[*].price products[*].inventory.count',
         'desc': 'The number of items currently in stock', 'answer': 'products[*].inventory.count'},
        {'schema': 'order.id order.total order.items[*].quantity',
         'desc': 'The shipping address of the order', 'answer': 'NO_ANSWER'},
    ]

    def build_fewshot_eval(ctx, question):
        messages = [{'role': 'system', 'content': SYSTEM_MSG}]
        for ex in FEWSHOT:
            messages.append({'role': 'user', 'content': f'Schema:\n{ex["schema"]}\n\nDescription: {ex["desc"]}'})
            messages.append({'role': 'assistant', 'content': ex['answer']})
        messages.append({'role': 'user', 'content': f'Schema:\n{ctx[:2500]}\n\nDescription: {question}'})
        return messages

    def normalize(s):
        return re.sub(r'\s+', ' ', s.strip().lower())

    def eval_qwen(data_path, ds_name, n_eval=QWEN_EVAL_SUBSET):
        print(f'\n{"="*60}\n  QWEN EVAL: {ds_name}\n{"="*60}')
        qa_pairs = read_qa_pairs_validation(data_path)
        # Stratified sample
        ans = [x for x in qa_pairs if x[2]['answer_start']]
        non = [x for x in qa_pairs if not x[2]['answer_start']]
        ratio = len(ans)/len(qa_pairs) if qa_pairs else 0.5
        n_ans = int(n_eval * ratio); n_non = n_eval - n_ans
        random.seed(42)
        random.shuffle(ans); random.shuffle(non)
        sampled = ans[:n_ans] + non[:n_non]
        random.shuffle(sampled)
        print(f'  {len(sampled)} samples ({min(n_ans,len(ans))} ans + {min(n_non,len(non))} non)')

        correct = ca = cn = 0
        ta = tn = 0
        t0 = time.time()

        for ctx, q, ans_obj in tqdm(sampled, desc='  Eval'):
            gt = ans_obj['text'][0] if ans_obj['text'] else ''
            is_ans = bool(ans_obj['answer_start'])

            messages = build_fewshot_eval(ctx, q)
            prompt = qwen_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = qwen_tok(prompt, return_tensors='pt', truncation=True, max_length=1536).to('cuda')
            with torch.no_grad():
                out = qwen_model.generate(
                    **inputs, max_new_tokens=32, do_sample=False,
                    pad_token_id=qwen_tok.pad_token_id,
                )
            pred = qwen_tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

            pn, gn = normalize(pred), normalize(gt)
            is_correct = False
            if is_ans:
                ta += 1
                if gn and (pn == gn or gn in pn or pn in gn):
                    is_correct = True; ca += 1
            else:
                tn += 1
                if pn in ['no_answer', 'no answer', '', 'n/a', 'none']:
                    is_correct = True; cn += 1
            if is_correct: correct += 1

        total = len(sampled)
        acc = correct/total
        acc_a = ca/ta if ta else 0
        acc_n = cn/tn if tn else 0
        elapsed = time.time() - t0
        print(f'\n  {elapsed/60:.1f} min | Acc@1={acc:.4f} (ans={acc_a:.4f} non={acc_n:.4f})')

        results = [{'rank':k,'correct':correct,'correct_answerable':ca,
                    'correct_non_answerable':cn,'total':total,
                    'total_answerable':ta,'total_non_answerable':tn,
                    'accuracy':acc,'accuracy_answerable':acc_a,
                    'accuracy_non_answerable':acc_n} for k in range(1,11)]
        return {'model':QWEN_NAME,'dataset':ds_name,'type':'qlora-fewshot',
                'total_samples':total,'total_answerable':ta,'total_non_answerable':tn,
                'elapsed_seconds':elapsed,'results':results}

    if not qwen_pm_done:
        r = eval_qwen(PM_PATH, 'parameter_matching')
        save_result(r, QWEN_NAME, 'pm')
        all_results_phase4.append(r)
    else:
        all_results_phase4.append(load_existing_result(qwen_pm_done))

    if not qwen_ed_done:
        r = eval_qwen(ED_PATH, 'endpoint_discovery')
        save_result(r, QWEN_NAME, 'ed')
        all_results_phase4.append(r)
    else:
        all_results_phase4.append(load_existing_result(qwen_ed_done))

    del qwen_model, qwen_tok; gc.collect(); torch.cuda.empty_cache()

print(f'\nTotal Phase 4 results: {len(all_results_phase4)}')

---
## Final Comparison: Phase 1 + 2 + 3 + 4

In [ ]:
# Cell 12: Final comparison table

# Baseline (hardcoded from Phase 1)
baseline_results = [
    {'model':'restberta-pm','dataset':'parameter_matching','type':'baseline',
     'total_samples':811004,
     'results':[{'rank':k,'accuracy':a,'accuracy_answerable':a_ans,'accuracy_non_answerable':a_non}
                for k,a,a_ans,a_non in [
                    (1,0.8529,0.7713,0.8684),
                    (3,0.9800,0.9244,0.9906),
                    (5,0.9897,0.9422,0.9987),
                    (10,0.9939,0.9663,0.9992),
                ] + [(k,0,0,0) for k in [2,4,6,7,8,9]]]},
]

combined = baseline_results + all_results_phase4

print(f'\n{"="*110}')
print(f'FULL COMPARISON — Phase 1 + Phase 4')
print(f'{"="*110}')
print(f'{"Model":<30} {"Type":<20} {"Dataset":<22} {"N":>7} {"Acc@1":>7} {"Acc@3":>7} {"Acc@5":>7} {"Acc@10":>7}')
print('-'*110)

BASELINE = 0.8529
best_pm_acc1 = 0
best_pm_model = ''

for r in combined:
    acc = {x['rank']: x.get('accuracy', 0) for x in r['results']}
    ds = r.get('dataset', '?')
    model = r.get('model', '?')
    typ = r.get('type', '?')
    n = r.get('total_samples', 0)
    marker = ''
    if 'parameter' in ds.lower() and acc.get(1, 0) > BASELINE:
        marker = '  ← BEATS BASELINE'
        if acc.get(1, 0) > best_pm_acc1:
            best_pm_acc1 = acc.get(1, 0)
            best_pm_model = model
    print(f"{model:<30} {typ:<20} {ds:<22} {n:>7} "
          f"{acc.get(1,0):>7.4f} {acc.get(3,0):>7.4f} {acc.get(5,0):>7.4f} {acc.get(10,0):>7.4f}{marker}")

print('='*110)
if best_pm_acc1 > 0:
    print(f'\n🎯 WINNER: {best_pm_model} — PM Acc@1 = {best_pm_acc1:.4f}')
    print(f'   Improvement over baseline: +{(best_pm_acc1-BASELINE)*100:.2f}%')
else:
    print(f'\n⚠ No model beat baseline {BASELINE:.4f}')
    print(f'   Best PM attempt: see table above')

# Save
with open(f'{RESULTS_DIR}/phase4_v4_final_comparison.json', 'w') as f:
    json.dump(combined, f, indent=2)
print(f'\nSaved to: {RESULTS_DIR}/phase4_v4_final_comparison.json')